# Quick validation of expert rating statistics etc.

In [8]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

In [9]:
raw_scores_df = pd.read_csv("../data/scores/PerformanceAssessmentData_finish_corrected_filtered.csv", sep=";")
df_ratings = pd.read_csv("../data/scores/merged_scores.csv")

len(raw_scores_df)

327

In [10]:
# participant number 8 should be excluded

unique = raw_scores_df['Participant Number'].unique()
sorted_unique = sorted(unique)
sorted_unique

[np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(17),
 np.int64(18),
 np.int64(19),
 np.int64(20),
 np.int64(21),
 np.int64(22),
 np.int64(23),
 np.int64(24),
 np.int64(25),
 np.int64(26),
 np.int64(27),
 np.int64(28),
 np.int64(29)]

### What is the mean error and standard deviation between expert scores (with respect to mean expert score)

**ref. paper: 4.6 ± 3.8**

**I find: 4.8 ± 3.5**     

**Why is this different !??**

there are 327 not 332 samples in my data.

also here participant 19 aparently performed all 3 tasks but only gets one expert rating for Ulna case

In [11]:
# add mean expert score per participant
mean_score_df = raw_scores_df.groupby(["Participant Number", "Case Type"])['QRS Overal'].mean().reset_index().rename(columns={"QRS Overal": "Mean Expert Score"})

raw_scores_df = raw_scores_df.merge(mean_score_df, on=["Participant Number", "Case Type"], how="left")
raw_scores_df.head()

mean_score_df.drop(index=51, inplace=True)
mean_score_df.describe()

,Participant Number,Mean Expert Score
count,83.000000,83.000000
mean,15.204819,49.938755
std,8.499296,9.364622
min,1.000000,33.750000
25%,8.000000,43.125000
50%,15.000000,49.250000
75%,22.500000,58.708333
max,29.000000,67.750000


In [12]:
# case specifi mean and std
case_mean_std_df = mean_score_df.groupby("Case Type")['Mean Expert Score'].agg(['mean', 'std']).reset_index()
case_mean_std_df.rename(columns={'mean': 'Case Mean Score', 'std': 'Case Std Dev'}, inplace=True)
case_mean_std_df

,Case Type,Case Mean Score,Case Std Dev
0,Ulna,52.354938,8.289487
1,malleolus,50.041667,10.202842
2,radius,47.505952,9.169582


In [13]:
err_series = raw_scores_df['QRS Overal'] - raw_scores_df['Mean Expert Score']
err_series.abs().describe()

count    327.000000
mean       4.833333
std        3.489540
min        0.000000
25%        2.125000
50%        4.250000
75%        7.000000
max       24.250000
dtype: float64

In [20]:
raw_scores_df['QRS Overal'].describe()

count    327.000000
mean      49.813456
std       11.076646
min       28.000000
25%       41.000000
50%       50.000000
75%       58.000000
max       70.000000
Name: QRS Overal, dtype: float64

In [21]:
# this is the participant that only performed 2 tasks, radius and fibula, not ulna 
raw_scores_df[raw_scores_df['Participant Number'] == 19]

,Zeitstempel,Participant number (XXX),Expert number,1.2 Time and Motion,1.3 Instrument Handling,1.4 Knowledge of Instruments,1.5 Flow of Procedure,1.6 Knowledge of Specific Procedure,1.7 Overall Performance,1.8 Quality of final product,...,"3.9 If not lag screw, achieves compression with plate",3.10 Safely/ effectively holds plate on bone with forceps/clamp/wire,3.11 Screws drilled using appropriate guide/ soft tissue protector,3.12 Plate appropriately secured to bone in reasonable position,3.13 all screws appropriate length,CFS Overall,Did you recognize the participant?,Participant Number,Case Type,Mean Expert Score
9,2024/05/08 12:22:15 PM OESZ,191,2,5,5,5,5,5,5,5,...,,0,0,0,0,0,No,19,radius,35.50
69,2024/06/10 5:26:10 PM OESZ,192,2,5,6,5,6,6,5,5,...,,1,1,1,1,9,No,19,malleolus,33.75
125,2024/07/30 11:25:00 PM OESZ,193,1,10,10,10,9,10,10,10,...,1,1,1,1,1,10,No,19,Ulna,69.00
173,2024/08/13 10:29:44 PM OESZ,192,1,4,4,4,4,4,4,5,...,,1,1,1,1,10,No,19,malleolus,33.75
211,2024/09/04 12:22:17 PM OESZ,192,4,5,5,5,6,5,5,5,...,,0,1,0,1,6,No,19,malleolus,33.75
232,2024/09/04 9:05:30 AM OESZ,191,4,6,6,7,5,6,6,6,...,,0,1,1,1,8,No,19,radius,35.50
287,2024/09/18 4:38:58 PM OESZ,191,3,5,5,6,6,5,5,5,...,,0,0,1,0,4,No,19,radius,35.50
291,2024/09/18 9:58:15 PM OESZ,191,1,4,4,4,4,4,4,4,...,,0,0,1,1,6,No,19,radius,35.50
316,2024/09/24 9:23:54 AM OESZ,192,3,5,5,5,4,5,4,4,...,,1,1,0,0,6,No,19,malleolus,33.75


Verify that these score are euqivalent to the ones used in analysis

In [ ]:
mean_score_df['Mean Expert Score'].describe()

count    83.000000
mean     49.938755
std       9.364622
min      33.750000
25%      43.125000
50%      49.250000
75%      58.708333
max      67.750000
Name: Mean Expert Score, dtype: float64

In [23]:
df_ratings[['Participant Number', 'Case_Number', 'QRS_Overal']]['QRS_Overal'].describe()

count    83.000000
mean     49.938755
std       9.364622
min      33.750000
25%      43.125000
50%      49.250000
75%      58.708333
max      67.750000
Name: QRS_Overal, dtype: float64

In [24]:
raw_scores_df['QRS Overal'].value_counts()

QRS Overal
56    36
49    21
63    15
42    15
35    11
45    10
47    10
37     9
39     9
62     9
70     9
36     9
51     8
28     8
38     8
69     8
58     8
53     8
50     8
60     7
40     7
52     7
44     6
46     6
33     6
48     5
64     5
55     5
66     5
61     5
43     5
54     5
57     4
59     4
65     4
41     4
34     3
31     3
68     3
32     3
30     2
29     2
67     2
Name: count, dtype: int64

## Intraclass Correlation Coefficient (ICC)

Mean Score (ICC2k = 0.91): By averaging them, you’ve captured the "consensus," which is much more stable and predictable for an algorithm.

"To ensure a highly reliable ground truth for the machine learning model, we utilized the mean score of 4 experts. The inter-rater reliability of this pooled mean was excellent (ICC2k = 0.91), whereas the reliability of a single rater was moderate (ICC2 = 0.71)."


In [25]:
import pingouin as pg

icc = pg.intraclass_corr(data=raw_scores_df, 
                        targets='Participant Number', 
                        raters='Expert number', 
                        ratings='QRS Overal',
                        nan_policy='omit')

print(icc)

    Type              Description       ICC          F  df1  df2  \
0   ICC1   Single raters absolute  0.709080  10.749471   26   81   
1   ICC2     Single random raters  0.712867  13.131083   26   78   
2   ICC3      Single fixed raters  0.752032  13.131083   26   78   
3  ICC1k  Average raters absolute  0.906972  10.749471   26   81   
4  ICC2k    Average random raters  0.908516  13.131083   26   78   
5  ICC3k     Average fixed raters  0.923845  13.131083   26   78   

           pval          CI95  
0  6.632287e-17  [0.56, 0.84]  
1  4.653834e-19  [0.54, 0.84]  
2  4.653834e-19  [0.61, 0.86]  
3  6.632287e-17  [0.83, 0.95]  
4  4.653834e-19  [0.83, 0.96]  
5  4.653834e-19  [0.86, 0.96]  


In [ ]:
# standard error of measurment (SEM) 
# This is the noise floor of the ground truth
sem_icc =  mean_score_df['Mean Expert Score'].std()  * np.sqrt(1 - icc[icc['Type'] == 'ICC2k']['ICC'].values[0])
sem_icc

np.float64(2.884763400817223)

**Comparison: Model MAE vs. SEM**

You can use the SEM to categorize your ML model’s performance:

MAE < 2.88: Your model is "Super-Human." It has effectively filtered out the noise in the expert ratings and is predicting the underlying signal better than the average error of the labels themselves.

MAE ≈ 2.88: Your model has reached "Expert Consensus Equivalence." It is as reliable as the panel of experts you used to create the ground truth.

MAE > 4.83 (Rater MAE): Your model is performing worse than a single random expert. This would suggest the model hasn't learned the "logic" of the scoring system yet.